# PayRevive degradation detection

Can PayRevive detect an issuer or gateway whose daily decline fraction rises above its recent baseline? This notebook replays initial payment outcomes for one verified synthetic dataset. The Python detector reads outcomes and cohort keys. The evaluator opens incident labels after scoring.

In [ ]:
import os
from pathlib import Path
from uuid import uuid4

import pandas as pd
from IPython.display import Image, display

from ml.analysis.degradation import run_detection

dataset = Path(os.environ.get("PAYREVIVE_NOTEBOOK_DATASET", "work/datasets/demo"))
output = Path(
    os.environ.get("PAYREVIVE_NOTEBOOK_OUTPUT", f"work/analysis/interactive-{uuid4().hex[:8]}")
)
report = run_detection(dataset, output, os.environ.get("PAYREVIVE_EXPECTED_MANIFEST"))
display(pd.Series(report["provenance"]["generator_config"], name="Generator configuration"))
print("Source manifest SHA-256", report["provenance"]["source_manifest_sha256"])

## What triggers an alert?

At each UTC day close, calculate failures divided by attempts for each group. Compare this with the pooled counts over its previous 14 calendar days. Both rules require 20 current attempts and 100 baseline attempts, giving them the same comparison windows.

The fixed rule fires at a decline rate of 35%. The rolling rule requires a rise of at least 10 percentage points and a standardized rise of at least 3. The standardization uses pooled binomial variance, so the same change tends to score higher with more observations. Treat the score as a heuristic. Repeated customers, simultaneous cohort checks, and variation by billing day complicate formal probability claims.

In [ ]:
evaluation = report["evaluation"]
comparison = pd.DataFrame.from_dict(evaluation["methods"], orient="index").drop(columns="incidents")
display(comparison)
print(f"Eligible windows {evaluation['eligible_windows']} / {evaluation['total_windows']}")
display(Image(filename=str(output / "timeline.png")))

## How the replay is judged

Incident recall counts injected incidents with an alert in the matching issuer or gateway bucket. Alert precision counts the share of alerted windows that overlap the corresponding incident. The false alert rate divides alerts outside matching incidents by eligible negative windows. Delay starts at incident onset and ends at the close of the first matching alerted bucket.

Issuer and gateway groups overlap. A gateway incident can show up in issuer outcomes, so a flagged issuer may show real sample degradation while counting as a false alert under strict target matching. Sparse buckets affect coverage, a prior incident can raise a later baseline, and day-close reporting can arrive after an incident has cleared. Review per-incident eligibility and delay alongside recall.

In [ ]:
incidents = [
    {"method": method, **incident}
    for method, metrics in evaluation["methods"].items()
    for incident in metrics["incidents"]
]
if incidents:
    display(pd.DataFrame(incidents))
else:
    print("This control dataset contains zero injected incidents.")